In [23]:
import os
print(os.listdir("/content/"))

['.config', 'processed', 'INFLUD24.csv', 'sample_data']


In [24]:
import pandas as pd

# Teste direto — sem variáveis intermediárias
df = pd.read_csv(
    "/content/INFLUD24.csv",
    sep=";",             # ponto e vírgula — padrão DATASUS
    encoding="latin-1",
    low_memory=False
)

print(f"✅ {df.shape[0]:,} linhas | {df.shape[1]} colunas")

✅ 279,184 linhas | 190 colunas


In [25]:
# Arquitetura: Camada de limpeza (2/5)
# Recebe os dados brutos da ingestão e os prepara para a camada de análise de SQL

import pandas as pd
import os

# Recarrega o arquivo bruto - ponto de entrada desta camada
df = pd.read_csv(
    "/content/INFLUD24.csv",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

print(f"✅ Dataset carregado: {df.shape[0]:,} linhas | {df.shape[1]} colunas")


✅ Dataset carregado: 279,184 linhas | 190 colunas


In [26]:
# LIMPEZA 1: Remove colunas completamente vazias
# Colunas 100% vazias não agregam valor á análise
# e ocupam memória desnecessariamente

# Conta colunas antes da limpeza
colunas_antes = df.shape[1]

# Remove apenas colunas onde todos os valores são nulos
df = df.dropna(axis=1, how="all")

# Conta colunas depois da limpeza
colunas_depois = df.shape[1]

# Imprime relatório
print(f"✅ Colunas removidas: {colunas_antes - colunas_depois}")
print(f"✅ Colunas restantes: {colunas_depois}")





✅ Colunas removidas: 8
✅ Colunas restantes: 182


In [27]:
# ============================================================
# LIMPEZA 2: Seleciona apenas colunas relevantes
# De 190 colunas, focamos nas que têm valor analítico
# real para saúde pública
# ============================================================

# Colunas selecionadas para análise de saúde pública
COLUNAS_RELEVANTES = [
    # Identificação e localização
    "DT_NOTIFIC",
    "SG_UF_NOT",
    "ID_MUNICIP",

    # Perfil do paciente
    "NU_IDADE_N",
    "CS_SEXO",
    "CS_RACA",
    "CS_ESCOL_N",

    # Informações clínicas
    "CLASSI_FIN",
    "EVOLUCAO",
    "UTI",
    "SUPORT_VEN",

    # Sintomas
    "FEBRE",
    "TOSSE",
    "DISPNEIA",
    "SATURACAO",

    # Vacinação
    "VACINA_COV",
]

# Filtra apenas colunas que realmente existem no dataset
# Essa linha estava faltando — ela cria a variável colunas_existentes
colunas_existentes = [c for c in COLUNAS_RELEVANTES if c in df.columns]

# Cria novo dataframe apenas com colunas relevantes
df_clean = df[colunas_existentes].copy()

print(f"✅ Colunas selecionadas: {df_clean.shape[1]}")
print(f"📊 Shape final: {df_clean.shape}")

✅ Colunas selecionadas: 16
📊 Shape final: (279184, 16)


In [28]:
# SAÍDA: Persiste o dado limpo para a próxima camada
# O arquivo processed é a entrada do notebook de análise SQL

import os # Importa o módulo os para manipular diretórios

# Caminho de sáida - pasta processed
OUTPUT_PATH = "processed/INFLUD24_clean.csv"

# Garante que o diretório de saída exista
output_dir = os.path.dirname(OUTPUT_PATH)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True) # Cria o diretório se ele não existir

# Salva sem o índice do pandas (não faz parte dos dados)
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Arquivo limpo salvo em: {OUTPUT_PATH}")
print(f"📊 Linhas: {df_clean.shape[0]:,}")
print(f"📋 Colunas: {df_clean.shape[1]}")
print(f"\n=== Amostra do dado limpo ===")
print(df_clean.head())

✅ Arquivo limpo salvo em: processed/INFLUD24_clean.csv
📊 Linhas: 279,184
📋 Colunas: 16

=== Amostra do dado limpo ===
   DT_NOTIFIC SG_UF_NOT           ID_MUNICIP  NU_IDADE_N CS_SEXO  CS_RACA  \
0  08/02/2023        RS  CAMPINA DAS MISSOES          81       M      1.0   
1  14/02/2023        SC        FLORIANOPOLIS          76       M      1.0   
2  27/02/2023        PR             CURITIBA           2       M      4.0   
3  16/02/2023        SP               CAJURU          18       F      1.0   
4  25/02/2023        RJ       RIO DE JANEIRO           8       M      4.0   

  CS_ESCOL_N  CLASSI_FIN EVOLUCAO  UTI  SUPORT_VEN  FEBRE  TOSSE  DISPNEIA  \
0          1         5.0        3  2.0         2.0    2.0    1.0       1.0   
1        NaN         4.0        1  2.0         3.0    NaN    NaN       NaN   
2          5         2.0        1  2.0         3.0    1.0    1.0       1.0   
3        NaN         5.0        1  NaN         3.0    1.0    1.0       NaN   
4          0         2.0     